## Dataset

This is the dataset from the original paper, showing, many different rows generated from other SR systems and their equations:

In [1]:
from egglog.exp.param_eq.original_results import load_original_results

rows = load_original_results()
rows

,dataset,raw_index,algorithm_raw,algorithm,algo_row,input_kind,n_params,n_rank,before_nodes,before_params,...,orig_expr,simpl_expr,orig_parsed_expr,simpl_parsed_expr,orig_parsed_n_params,simpl_parsed_n_params,before_rank_difference,after_rank_difference,before_parsed_rank_difference,after_parsed_rank_difference
0,kotanchek,0,Bingo,Bingo,1,original,4.0,4.0,36.0,7.0,...,0.11064466475608078 + (-0.010036545250561161)*...,((0.11064466475608076) + (((x0) + (x1)) * (-2....,0.11064466475608078 + -0.010036545250561161 * ...,0.11064466475608076 + (x0 + x1) * -0.020073090...,4.0,5.0,3.0,1.0,0.0,1.0
1,kotanchek,0,Bingo,Bingo,1,sympy,4.0,4.0,47.0,8.0,...,(1.42614439569855*x1*(x0**2 + 0.02252279904556...,None,(1.42614439569855 * x1 * (x0 ** 2.0 + 0.022522...,None,5.0,NaN,4.0,2.0,1.0,NaN
2,kotanchek,2,Bingo,Bingo,2,original,6.0,5.0,24.0,6.0,...,(0.12833632981483378)*((x0 + (0.57174672264895...,(0.12833632981483378) * (((x0) + ((0.571746722...,0.12833632981483378 * ((x0 + 0.571746722648954...,0.12833632981483378 * ((x0 + 0.571746722648954...,11.0,5.0,1.0,0.0,6.0,0.0
3,kotanchek,2,Bingo,Bingo,2,sympy,6.0,5.0,29.0,8.0,...,(-0.12833632981483378*x0 - 0.07337587596842649...,None,(-0.12833632981483378 * x0 - 0.073375875968426...,None,5.0,NaN,3.0,1.0,0.0,NaN
4,kotanchek,3,Bingo,Bingo,3,original,4.0,4.0,22.0,4.0,...,-0.01768299095097286 + (-0.002127834025158428)...,(-1.768299095097286e-2) + ((-0.799049025992938...,-0.01768299095097286 + -0.002127834025158428 *...,-0.01768299095097286 + -0.799049025992938 * (x...,5.0,4.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
709,pagie,207,SRjl,PySR,28,sympy,15.0,8.0,36.0,10.0,...,0.19651888277923811*log(x1**2 + 0.029674209579...,None,0.1965188827792381 * log(x1 ** 2.0 + 0.0296742...,None,9.0,NaN,2.0,0.0,1.0,NaN
710,pagie,208,SRjl,PySR,29,original,12.0,5.0,35.0,12.0,...,(((((3.236280732539138 - (exp(((((x0 * 0.87118...,(((((3.2712151658590822) - (exp((((x0) * (x0))...,(3.236280732539138 - (exp((x0 * 0.871186294814...,(3.2712151658590822 - exp((x0 * x0 + -1.160457...,16.0,9.0,7.0,4.0,11.0,4.0
711,pagie,208,SRjl,PySR,29,sympy,12.0,5.0,23.0,7.0,...,1.9889982996591087 - 1.0486873457854022*exp(-0...,None,1.9889982996591087 - 1.0486873457854022 * exp(...,None,5.0,NaN,2.0,0.0,0.0,NaN
712,pagie,209,SRjl,PySR,30,original,7.0,6.0,28.0,7.0,...,(((((2.446632681660071 / exp((x0 / 0.784937509...,(-0.16360691420886442) * (((((2.44663268166007...,(2.446632681660071 / exp(x0 / 0.78493750912777...,-0.16360691420886442 * (2.446632681660071 / ex...,7.0,6.0,1.0,0.0,1.0,0.0


## Target Row

First we find a row we are working with:

In [2]:
row = rows[rows["orig_parsed_expr"].str.startswith("(-0.0218")]
print(row)

    dataset  raw_index algorithm_raw algorithm  algo_row input_kind  n_params  \
623   pagie        164           SBP       SBP        14      sympy       6.0   

     n_rank  before_nodes  before_params  ...  \
623     5.0          31.0            7.0  ...   

                                             orig_expr  simpl_expr  \
623  (-0.021803393734787*x1*(2*x1 - 10.429) - 0.021...        None   

                                      orig_parsed_expr simpl_parsed_expr  \
623  (-0.021803393734787 * x1 * (2.0 * x1 - 10.429)...              None   

    orig_parsed_n_params simpl_parsed_n_params  before_rank_difference  \
623                  5.0                   NaN                     2.0   

     after_rank_difference  before_parsed_rank_difference  \
623                    0.0                            0.0   

     after_parsed_rank_difference  
623                           NaN  

[1 rows x 22 columns]


## Parsing
_Parsing is first done into a binary form, then into containers. Different cost models during extraction preference minimizing floating point constants and also replicate the cost on the container side of it turned into a binary expression_

We can look at the source expression we start with, which we want to simplify be removing the number of floats:

In [3]:
from egglog import *
from egglog.exp.param_eq.domain import binary_to_containers
from egglog.exp.param_eq.pipeline import *

print(expr_str := row["orig_parsed_expr"].iloc[0])

(-0.021803393734787 * x1 * (2.0 * x1 - 10.429) - 0.021803393734787 * x1 + 0.001352 * exp(x0 * (x0 - 2.0)) + 1.38875269916455e-07) * exp(x0 * (2.0 - x0))


We can parse this into a binary egglog expression tree, and get the "cost" using our custom cost model that separates number of floating point constants from other ops, and always chooses the one with less floats first:

In [4]:
expr_binary, cost_binary = EGraph().extract(parse_expression(expr_str), include_cost=True, cost_model=param_cost_model)
print(cost_binary, expr_binary)

ParamCost(5, 26) (
    Num(-0.021803393734787) * Num.var("x1") * (Num(2.0) * Num.var("x1") - Num(10.429))
    - Num(0.021803393734787) * Num.var("x1")
    + Num(0.001352) * exp(Num.var("x0") * (Num.var("x0") - Num(2.0)))
    + Num(1.38875269916455e-07)
) * exp(Num.var("x0") * (Num(2.0) - Num.var("x0")))


Then we can compare that to the container representation, which has a similar cost, even though it's store differently:

In [ ]:
EGraph().extract(binary_to_containers(parse_expression("x ^ 2  / y")))

exp(Num.var("x") ** Num(2.0) * Num.var("y") - Num.var("z")) + Num(2.0)

In [5]:
expr_container, cost_container = EGraph().extract(
    binary_to_containers(expr_binary), include_cost=True, cost_model=container_cost_model
)
(cost_container, expr_container)

(ParamCost(5, 28),
 polynomial(
     {
         {
             polynomial(
                 {
                     {Num(-0.021803393734787): 1, Num.var("x1"): 1, polynomial({{Num.var("x1"): 1, Num(2.0): 1}: 1.0, {Num(10.429): 1}: -1.0}): 1}: 1.0,
                     {Num.var("x1"): 1, Num(0.021803393734787): 1}: -1.0,
                     {Num(0.001352): 1, exp(polynomial({{Num.var("x0"): 1, polynomial({{Num(2.0): 1}: -1.0, {Num.var("x0"): 1}: 1.0}): 1}: 1.0})): 1}: 1.0,
                     {Num(1.38875269916455e-07): 1}: 1.0,
                 }
             ): 1,
             exp(polynomial({{Num.var("x0"): 1, polynomial({{Num(2.0): 1}: 1.0, {Num.var("x0"): 1}: -1.0}): 1}: 1.0})): 1,
         }: 1.0
     }
 ))

The reason why the cost is slightly different is because the container is transformed to binary expressions in a way that moves around some things and turns a `x - a` into a `-1 * x + a`:

In [6]:
containers_to_binary(expr_container)

(
    Num(-0.021803393734787) * Num.var("x1") * (Num.var("x1") * Num(2.0) - Num(10.429))
    - Num.var("x1") * Num(0.021803393734787)
    + Num(0.001352) * exp(Num.var("x0") * (Num(-1.0) * Num(2.0) + Num.var("x0")))
    + Num(1.38875269916455e-07)
) * exp(Num.var("x0") * (Num(2.0) - Num.var("x0")))

## Results
_In this example, both binary and containers reduce the params to minimal number, but the binary one takes a lot longer and has a larger e-graph_:
### Binary

In [19]:
import time

start_time = time.perf_counter()

egraph = EGraph(save_egglog_string=True)
egraph.register(expr_binary)
egraph.run(
    binary_analysis_schedule
    + (run(binary_rewrite_ruleset, scheduler=rewrite_scheduler) + binary_analysis_schedule) * 30
)
expr_binary_simplified, expr_binary_simplified_cost = egraph.extract(
    expr_binary, include_cost=True, cost_model=param_cost_model
)
print(f"Time taken: {time.perf_counter() - start_time:.2f} seconds")
print(f"EGraph size: {sum(s for _, s in egraph.all_function_sizes())}")
print(f"Extracted cost: {expr_binary_simplified_cost}")
binary_egglog_program = egraph.as_egglog_string
expr_binary_simplified

Time taken: 0.53 seconds
EGraph size: 5768
Extracted cost: ParamCost(4, 21)


(Num(-0.043606787469574) * (Num.var("x1") * (Num(-4.7145) + Num.var("x1"))) + Num(0.001352) * exp(Num.var("x0") * (Num.var("x0") + Num(-2.0))) + Num(1.38875269916455e-07)) * exp(
    Num.var("x0") * (Num(2.0) - Num.var("x0"))
)

### Containers

In [20]:
start_time = time.perf_counter()
egraph = EGraph(save_egglog_string=True)
egraph.register(expr_container)
egraph.run(
    containers_analysis_schedule
    + (run(container_rewrite_ruleset, scheduler=rewrite_scheduler) + containers_analysis_schedule) * 30
)
expr_container_simplified, expr_container_simplified_cost = egraph.extract(
    expr_container, include_cost=True, cost_model=container_cost_model
)
print(f"Time taken: {time.perf_counter() - start_time:.2f} seconds")
print(f"EGraph size: {sum(s for _, s in egraph.all_function_sizes())}")
print(f"Extracted cost: {expr_container_simplified_cost}")
container_egglog_program = egraph.as_egglog_string
expr_container_simplified

Time taken: 0.09 seconds
EGraph size: 58
Extracted cost: ParamCost(4, 21)


polynomial(
    {
        {
            polynomial(
                {
                    {}: 1.38875269916455e-07,
                    {exp(polynomial({{Num.var("x0"): 1, polynomial({{}: -2.0, {Num.var("x0"): 1}: 1.0}): 1}: 1.0})): 1}: 0.001352,
                    {Num.var("x1"): 1, polynomial({{polynomial({{}: -4.7145, {Num.var("x1"): 1}: 1.0}): 1}: -0.043606787469574}): 1}: 1.0,
                }
            ): 1,
            exp(polynomial({{Num.var("x0"): 1, polynomial({{}: 2.0, {Num.var("x0"): 1}: -1.0}): 1}: 1.0})): 1,
        }: 1.0
    }
)

In [ ]:
containers_to_binary(expr_container_simplified)

## Egglog Lowering

Both of these execute as pure egglog programs, except for the custom cost models which are written in Python, both for containers and binary expressions. The rest though all works on the fork of egglog-experimental. The Python is just sugar.

In [21]:
print(binary_egglog_program)

(sort Num)
(constructor Num___init__ (f64) Num)
(let $__expr_0 (Num___init__ 2.0))
(constructor Num_var (String) Num)
(let $__expr_1 (Num_var "x1"))
(let $__expr_2 (Num_var "x0"))
(constructor Num___mul__ (Num Num) Num)
(constructor Num___add__ (Num Num) Num)
(constructor Num___sub__ (Num Num) Num)
(constructor egglog_exp_param_eq_domain_exp (Num) Num)
(Num___mul__ (Num___add__ (Num___add__ (Num___sub__ (Num___mul__ (Num___mul__ (Num___init__ -0.021803393734787) $__expr_1) (Num___sub__ (Num___mul__ $__expr_0 $__expr_1) (Num___init__ 10.429))) (Num___mul__ (Num___init__ 0.021803393734787) $__expr_1)) (Num___mul__ (Num___init__ 0.001352) (egglog_exp_param_eq_domain_exp (Num___mul__ $__expr_2 (Num___sub__ $__expr_2 $__expr_0))))) (Num___init__ 0.000000138875269916455)) (egglog_exp_param_eq_domain_exp (Num___mul__ $__expr_2 (Num___sub__ $__expr_0 $__expr_2))))
(ruleset egglog.exp.param_eq.pipeline.shared_analysis_rules)
(rewrite (egglog_exp_param_eq_domain_exp (Num___init__ _a)) (Num___ini

In [22]:
print(container_egglog_program)

(sort Num)
(constructor Num___init__ (f64) Num)
(let $__expr_0 (Num___init__ 2.0))
(sort Map[Num,BigRat] (Map Num BigRat))
(sort Map[Map[Num,BigRat],f64] (Map Map[Num,BigRat] f64))
(constructor polynomial (Map[Map[Num,BigRat],f64]) Num)
(constructor Num_var (String) Num)
(let $__expr_1 (Num_var "x1"))
(constructor egglog_exp_param_eq_domain_exp (Num) Num)
(polynomial (map-insert (map-empty) (map-insert (map-insert (map-empty) (polynomial (map-insert (map-insert (map-insert (map-insert (map-empty) (map-insert (map-insert (map-insert (map-empty) (Num___init__ -0.021803393734787) (bigrat (from-string "1") (from-string "1"))) $__expr_1 (bigrat (from-string "1") (from-string "1"))) (polynomial (map-insert (map-insert (map-empty) (map-insert (map-insert (map-empty) $__expr_1 (bigrat (from-string "1") (from-string "1"))) $__expr_0 (bigrat (from-string "1") (from-string "1"))) 1.0) (map-insert (map-empty) (Num___init__ 10.429) (bigrat (from-string "1") (from-string "1"))) -1.0)) (bigrat (from-